# scConcept Tutorial: Hierarchical Concept Refinement of Single-Cell Topics

This tutorial demonstrates how to perform **hierarchical concept refinement** using **scConcept**, 
given precomputed topic gene lists and first-level concepts.

---

## Prerequisites

Before running this tutorial, you should have already:

1. Trained a topic model on your single-cell RNA-seq dataset using **ECRTM**  
2. Generated **first-level concepts** from topic gene lists (see Tutorial 1)

---

## What this tutorial does

Starting from first-level concepts, this tutorial walks through:

1. **Assigning cells to first-level concepts** based on gene expression  
2. **Detecting heterogeneous concepts** using an expression-based splitting criterion  
3. **Refining concepts into sub-concepts** using a large language model (LLM)  
4. **Assigning cells to second-level (refined) concepts**  
5. **Performing hierarchical assignment** combining both levels  
6. **Evaluating refinement quality** using clustering metrics (ARI, NMI)  

---

## Expected outcome

By the end of this tutorial, you will obtain:

- A hierarchy of **coarse-to-fine biological concepts**  
- **Cell-level hierarchical annotations** (level-1 and level-2)  
- Quantitative evaluation of **refinement effectiveness**  
- Improved alignment between **concept structure and underlying cell heterogeneity**

---

## Note

This tutorial assumes that first-level concept distillation has already been completed.  
Concept refinement is performed only when sufficient evidence of heterogeneity is detected.

---

## Overview

This tutorial extends scConcept from flat concept discovery to **hierarchical concept modeling**,  
enabling the decomposition of broad biological programs into **more specific and interpretable sub-programs**.

In [ ]:
import numpy as np
from types import SimpleNamespace
from pathlib import Path
import sys

# =========================================================
# Project setup for the tutorial notebook
# =========================================================

PROJECT_ROOT = Path.cwd().parent
ECRTM_PATH = PROJECT_ROOT / "ECRTM"
sys.path.append(str(ECRTM_PATH))

from singlecell_dataset import SingleCellDataset

# =========================================================
# User-defined configuration
# =========================================================

dataset_name = "Pancreas_10k"
data_dir = PROJECT_ROOT / "Datasets"
batch_size = 2048

# =========================================================
# Build configuration namespace
# =========================================================

config = SimpleNamespace(
    dataset=dataset_name,
    batch_size=batch_size,
    data_dir=str(data_dir),
)

# =========================================================
# Load dataset
# =========================================================

dataset = SingleCellDataset(
    dataset_name=config.dataset,
    batch_size=config.batch_size,
    data_dir=config.data_dir,
)

train_matrix = dataset.train_data
test_matrix = dataset.test_data
train_labels = dataset.train_labels
test_labels = dataset.test_labels
gene_names = dataset.gene_names
num_genes = dataset.n_genes

# =========================================================
# Sanity check
# =========================================================

print(f"Train data shape: {train_matrix.shape}")
print(f"Test data shape: {test_matrix.shape}")
print(f"Train labels shape: {np.shape(train_labels)}")
print(f"Test labels shape: {np.shape(test_labels)}")
print(f"Number of genes: {num_genes}")
print(f"First 10 genes: {gene_names[:10]}")

===> Train size: 121916
===> Test size: 121916
===> Number of genes: 10000
===> Avg expression per cell: 422.345
===> Number of labels: 9
Train data shape: torch.Size([121916, 10000])
Test data shape: torch.Size([121916, 10000])
Train labels shape: (1, 121916)
Test labels shape: (1, 121916)
Number of genes: 10000
First 10 genes: ['7SK', '7SK.2', 'A2M', 'AAED1', 'AASS', 'AATK', 'AATK-AS1', 'ABAT', 'ABCA1', 'ABCA10']


In [ ]:
from pathlib import Path

# =========================================================
# Function: load topic gene lists
# =========================================================

def load_topic_genes(topic_file):
    """
    Load topic gene lists from a text file.

    Each line corresponds to one topic, where genes are separated by spaces.

    Parameters
    ----------
    topic_file : str or Path
        Path to the topic gene file.

    Returns
    -------
    list[list[str]]
        A list of topics, where each topic is a list of gene names.
    """
    topic_gene_lists = []

    with open(topic_file, "r", encoding="utf-8") as f:
        for line in f:
            genes = line.strip().split()
            topic_gene_lists.append(genes)

    return topic_gene_lists


# =========================================================
# Automatically construct topic file path
# =========================================================

num_topics = 50
seed = 1
epoch = 500

topic_dir = (
    PROJECT_ROOT
    / "ECRTM"
    / "output"
    / "topics"
    / f"{dataset_name}_K{num_topics}_seed{seed}"
)

topic_file = topic_dir / f"epoch{epoch}_top_genes.txt"

# Check file existence
if not topic_file.exists():
    raise FileNotFoundError(f"Topic file not found: {topic_file}")

# =========================================================
# Load topic gene lists
# =========================================================

topic_gene_lists = load_topic_genes(topic_file)

# =========================================================
# Sanity check
# =========================================================

print(f"Topic file: {topic_file}")
print(f"Number of topics: {len(topic_gene_lists)}")
print(f"Top 20 genes in Topic 0: {topic_gene_lists[0][:20]}")

In [ ]:
import json
from pathlib import Path
from openai import OpenAI

# =========================================================
# Convert topic gene lists into JSON payload
# =========================================================
# Each topic is represented by:
# - topic_id: topic index
# - genes: top genes of that topic
#
# This prepares the input for first-level concept distillation.

topics_payload = [
    {
        "topic_id": f"topic_{topic_id}",
        "genes": topic_gene_lists[topic_id]
    }
    for topic_id in range(len(topic_gene_lists))
]

topics_json_str = json.dumps({"topics": topics_payload}, indent=2)


# =========================================================
# Build the topic distillation prompt (DO NOT MODIFY)
# =========================================================
# The following prompt is used to generate first-level concepts
# directly from topic gene lists.

topic_distillation_prompt = f"""
You are given topics derived from a neural topic model on single-cell RNA-seq.

Each topic includes ONLY:
- A list of the top 100 genes for that topic,
  sorted from highest weight to lowest weight in the model.

Here are all topics (JSON list):
{topics_json_str}

TASK:
You must perform *biological topic distillation* by analyzing ONLY the top-100 gene lists.

Specifically:
1. Identify topics whose top genes indicate highly similar biological pathways or cell states.
2. Merge such topics into unified, biologically coherent concepts.
3. Remove topics that are:
   - too vague (gene list does not form a coherent module),
   - too narrow (dominated by a single outlier gene),
   - biologically uninterpretable given the provided gene patterns.

For each FINAL biological concept, output:
- "name": a concise 2–4 word biological label.
      * Avoid generic labels like "General Regulation".
      * Prefer pathway- or state-specific terms (e.g., "Cell Cycle Progression").
- "genes": EXACTLY 100 representative genes.
      * MUST be chosen from the union of gene lists of the merged source topics.
      * MUST reflect a coherent pathway or co-expression program.
      * MUST respect the original sorted importance: higher-ranked genes are preferred.
      * DO NOT invent gene names.
- "source_topics": list of topic IDs merged into this concept.

OUTPUT FORMAT (STRICT):
Respond ONLY with a valid JSON object in EXACTLY this form:

{{
  "concepts": [
    {{
      "name": "Example Name",
      "genes": ["GENE1", "GENE2", "GENE3", ..., "GENE100"],
      "source_topics": ["topic_0", "topic_3"]
    }}
  ]
}}

ABSOLUTE RULES:
- DO NOT output anything outside the JSON.
- DO NOT add code fences (such as ```json).
- DO NOT invent genes.
- DO NOT reorder the topics_json_str; use it as provided.
- The final genes must reflect the most informative subset of the top 100 genes per topic.
"""


# =========================================================
# Call the OpenAI API to generate first-level concepts
# =========================================================
# Make sure your OPENAI_API_KEY environment variable is set.

client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-5",
    seed=1,
    messages=[
        {"role": "system", "content": "You are an expert in single-cell biology."},
        {"role": "user", "content": topic_distillation_prompt}
    ]
)

response_text = completion.choices[0].message.content


# =========================================================
# Parse the LLM output
# =========================================================

concept_result = json.loads(response_text)
level1_concepts = concept_result["concepts"]

print(f"Number of first-level concepts generated: {len(level1_concepts)}")
print(f"First concept name: {level1_concepts[0]['name']}")


# =========================================================
# Save the generated first-level concepts
# =========================================================
# Output path:
# Results/<dataset_name>/gpt5_<dataset_name>_level1_concepts.json

results_dir = PROJECT_ROOT / "Results" / dataset_name
results_dir.mkdir(parents=True, exist_ok=True)

output_file = results_dir / f"gpt5_{dataset_name}_level1_concepts.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(level1_concepts, f, indent=2, ensure_ascii=False)

print(f"Saved first-level concepts to: {output_file}")

In [5]:
import numpy as np

# =========================================================
# Utility functions for first-level concept assignment
# =========================================================

def build_gene_index(gene_names):
    """
    Create a mapping from gene name to column index.

    Parameters
    ----------
    gene_names : list-like
        Ordered list of gene names.

    Returns
    -------
    dict
        Dictionary mapping gene name to column index.
    """
    return {str(gene): idx for idx, gene in enumerate(gene_names)}


def truncate_concept_genes(concepts, top_k):
    """
    Keep only the top-k genes for each concept.

    Parameters
    ----------
    concepts : list[dict]
        List of first-level concept dictionaries.
    top_k : int
        Number of top genes to retain.

    Returns
    -------
    list[dict]
        Updated concept list with truncated gene sets.
    """
    truncated_concepts = []

    for concept in concepts:
        truncated_concepts.append(
            {
                "name": concept["name"],
                "genes": concept["genes"][:top_k],
                "source_topics": concept.get("source_topics", [])
            }
        )

    return truncated_concepts


def assign_cells_to_concepts_by_zscore(concepts, expression_matrix, gene_to_index):
    """
    Assign each cell to the concept with the highest average marker z-score.

    Parameters
    ----------
    concepts : list[dict]
        List of concept dictionaries. Each concept should contain:
        - "name"
        - "genes"
    expression_matrix : torch.Tensor
        Cell-by-gene expression matrix of shape (num_cells, num_genes).
    gene_to_index : dict
        Mapping from gene name to column index.

    Returns
    -------
    score_matrix : np.ndarray
        Matrix of shape (num_cells, num_concepts).
    concept_names : list[str]
        Names of all concepts.
    predicted_labels : list[str]
        Predicted first-level concept label for each cell.
    """
    expression_array = expression_matrix.detach().cpu().numpy().astype(float)

    gene_mean = expression_array.mean(axis=0, keepdims=True)
    gene_std = expression_array.std(axis=0, keepdims=True) + 1e-6
    expression_zscore = (expression_array - gene_mean) / gene_std

    concept_names = [concept["name"] for concept in concepts]
    num_cells = expression_array.shape[0]
    num_concepts = len(concepts)

    score_matrix = np.zeros((num_cells, num_concepts), dtype=float)

    for concept_index, concept in enumerate(concepts):
        concept_genes = concept["genes"]

        gene_indices = [
            gene_to_index[gene]
            for gene in concept_genes
            if gene in gene_to_index
        ]

        if len(gene_indices) == 0:
            continue

        score_matrix[:, concept_index] = expression_zscore[:, gene_indices].mean(axis=1)

    predicted_indices = np.argmax(score_matrix, axis=1)
    predicted_labels = [concept_names[idx] for idx in predicted_indices]

    return score_matrix, concept_names, predicted_labels


# =========================================================
# Assign cells to first-level concepts
# =========================================================

gene_to_index = build_gene_index(gene_names)

top_k = 100
top_k_level1_concepts = truncate_concept_genes(
    level1_concepts,
    top_k=top_k
)

level1_score_matrix, level1_concept_names, predicted_level1_concepts = (
    assign_cells_to_concepts_by_zscore(
        top_k_level1_concepts,
        test_matrix,
        gene_to_index
    )
)

# =========================================================
# Sanity check
# =========================================================

print(f"Level-1 score matrix shape: {level1_score_matrix.shape}")
print(f"Number of level-1 concepts: {len(level1_concept_names)}")
print(f"Level-1 concept names: {level1_concept_names}")
print(f"Example predicted level-1 concepts: {predicted_level1_concepts[:10]}")

Level-1 score matrix shape: (121916, 10)
Number of level-1 concepts: 10
Level-1 concept names: ['Pancreatic acinar secretion', 'Islet endocrine hormones', 'Epithelial inflammatory response', 'Acute phase response', 'Endothelial angiogenesis', 'Fibroblast ECM remodeling', 'Mucosal secretory epithelium', 'Gastric acid secretion', 'Cytotoxic T/NK activation', 'CFTR ductal epithelium']
Example predicted level-1 concepts: ['Islet endocrine hormones', 'Islet endocrine hormones', 'Islet endocrine hormones', 'Islet endocrine hormones', 'Islet endocrine hormones', 'Islet endocrine hormones', 'Islet endocrine hormones', 'Islet endocrine hormones', 'Islet endocrine hormones', 'Islet endocrine hormones']


In [6]:
import json
import numpy as np
import torch
from openai import OpenAI
from sklearn.cluster import KMeans

# =========================================================
# Hierarchical prompt (DO NOT MODIFY)
# =========================================================

HIERARCHICAL_PROMPT_TEMPLATE_FORCE_SPLIT = """
You are given ONE biological concept derived from a neural topic model on single-cell RNA-seq.

IMPORTANT:
This concept has been determined to be HETEROGENEOUS and contains multiple
distinct biological programs. Your task is to split it into meaningful sub-concepts.

The input concept is a JSON object with:
- "name": concept name
- "genes": EXACTLY 100 genes representing this concept (gene program)

Here is the input concept (JSON):
{concept_json}

TASK:
Split this concept into biologically meaningful sub-concepts.

A biological program may also correspond to a distinct cell identity
defined by characteristic combinations of marker genes.
Prefer splits that separate mutually exclusive marker gene groups
indicative of different cell identities.
If mutually exclusive marker gene groups defining distinct cell identities
are present, prioritize identity-based splits over shared functional or
pathway-level programs.

------------------------------------------------------------
SPLIT REQUIREMENTS (MANDATORY)
------------------------------------------------------------

- Produce EXACTLY 2 to 4 sub-concepts.
- EACH sub-concept must represent a SINGLE, coherent biological program.
- Sub-concepts must be generalizable and interpretable.

For EACH sub-concept:
- "name": concise 2–4 word biological label.
- "description": a 10–30 word natural language description
      * Summarize the biological meaning of this sub-concept.
      * Describe the core pathway, cellular program, or functional state.
      * Must be consistent with the sub-concept name and gene list.
      * Do NOT mention modeling details or the parent concept.
- "genes": select between 5 and 20 genes.
- ALL genes MUST be chosen from the parent concept's gene list.
- DO NOT invent genes.
- DO NOT add genes not present in the parent list.

------------------------------------------------------------
OUTPUT FORMAT (STRICT JSON ONLY)
------------------------------------------------------------

Respond with EXACTLY one JSON object in the following format:

{{
  "concept_name": "<parent_concept_name>",
  "split": true,
  "sub_concepts": [
    {{
      "name": "<sub_concept_name>",
      "description": "A concise description summarizing the biological program represented by this sub-concept.",
      "genes": ["GENE1", "GENE2", ...]
    }}
  ]
}}

OUTPUT RULES:
- "split" MUST be true.
- "sub_concepts" MUST contain 2-4 items.
- Each "genes" list length MUST be between 5 and 20 (inclusive).
- Every gene MUST appear in the parent concept gene list.
- Respond ONLY with valid JSON.
- Do NOT include explanations, comments, or markdown.
"""

In [7]:
# =========================================================
# Decide whether a concept should be split (PCT-style test)
# =========================================================

def pct_should_split_concept_expr_matrix(
    X,
    vocab,
    concepts,
    score_old,
    concept_idx,
    tau_frac=0.5,
    min_cells=30,
    min_leaf=20,
    min_impurity_reduction=0.1,
    random_state=0,
):
    """
    Expression-based split decision (no labels required).

    This function checks whether a concept contains heterogeneous
    expression patterns and should be further split.
    """

    info = {"concept_idx": int(concept_idx)}

    gene_to_idx = {g: i for i, g in enumerate(vocab)}

    # ---- select high-confidence cells ----
    s_all = score_old[:, concept_idx]
    tau = tau_frac * float(np.max(s_all))
    cell_idx = np.where(s_all > tau)[0]

    if len(cell_idx) < min_cells:
        return False, {"reason": "too_few_cells"}

    # ---- select concept genes ----
    concept_genes = concepts[concept_idx]["genes"]
    gene_indices = [gene_to_idx[g] for g in concept_genes if g in gene_to_idx]

    if len(gene_indices) < 10:
        return False, {"reason": "too_few_genes"}

    # ---- extract expression ----
    if hasattr(X, "detach"):
        X_sub = X[cell_idx][:, gene_indices].detach().cpu().numpy()
    else:
        X_sub = X[np.ix_(cell_idx, gene_indices)]

    # ---- compute impurity ----
    mu = X_sub.mean(axis=0, keepdims=True)
    parent_sse = np.sum((X_sub - mu) ** 2)

    if parent_sse < 1e-8:
        return False, {"reason": "near_zero_variance"}

    # ---- KMeans split ----
    km = KMeans(n_clusters=2, random_state=random_state, n_init=10)
    labels = km.fit_predict(X_sub)

    idx0 = labels == 0
    idx1 = labels == 1

    if min(idx0.sum(), idx1.sum()) < min_leaf:
        return False, {"reason": "small_cluster"}

    mu0 = X_sub[idx0].mean(axis=0, keepdims=True)
    mu1 = X_sub[idx1].mean(axis=0, keepdims=True)

    child_sse = np.sum((X_sub[idx0] - mu0) ** 2) + np.sum((X_sub[idx1] - mu1) ** 2)

    reduction = (parent_sse - child_sse) / parent_sse

    if reduction < min_impurity_reduction:
        return False, {"reason": "low_gain"}

    return True, {"reason": "PASS"}

In [ ]:
# =========================================================
# Split one concept using GPT
# =========================================================

client = OpenAI(api_key="")

def refine_concept_with_gpt(concept):
    """
    Split one concept into sub-concepts using GPT.
    """

    concept_json_str = json.dumps(concept, ensure_ascii=False)

    prompt = HIERARCHICAL_PROMPT_TEMPLATE_FORCE_SPLIT.format(
        concept_json=concept_json_str
    )

    resp = client.chat.completions.create(
        model="gpt-5",
        seed=1,
        messages=[
            {"role": "system", "content": "Return ONLY valid JSON."},
            {"role": "user", "content": prompt}
        ]
    )

    output_text = resp.choices[0].message.content.strip()

    if output_text.startswith("```"):
        output_text = output_text.strip("`").replace("json", "").strip()

    return json.loads(output_text)

In [9]:
# =========================================================
# Refine all first-level concepts into second-level concepts
# =========================================================

def refine_all_concepts(
    X,
    vocab,
    concepts,
    score_old
):
    """
    Perform hierarchical refinement for all concepts.

    Returns:
        refined_results: list of refined concept structures
    """

    refined_results = []

    for idx, concept in enumerate(concepts):
        print(f"\n=== Processing concept {idx} ===")

        should_split, info = pct_should_split_concept_expr_matrix(
            X=X,
            vocab=vocab,
            concepts=concepts,
            score_old=score_old,
            concept_idx=idx,
        )

        if should_split:
            print("→ SPLIT")
            refined = refine_concept_with_gpt(concept)
        else:
            print("→ KEEP")
            refined = {
                "sub_concepts": [
                    {
                        "name": concept["name"],
                        "genes": concept["genes"]
                    }
                ]
            }

        refined_results.append(refined)

    return refined_results

In [10]:
# =========================================================
# Run hierarchical concept refinement
# =========================================================

refined_level2_concepts = refine_all_concepts(
    X=test_matrix,
    vocab=gene_names,
    concepts=level1_concepts,
    score_old=level1_score_matrix
)

print(f"Number of refined concepts: {len(refined_level2_concepts)}")
print("Example refined concept:")
print(refined_level2_concepts[0])

# =========================================================
# Save the refined second-level concepts
# =========================================================
# Output path:
# Results/<dataset_name>/gpt5_<dataset_name>_level2_concepts.json

results_dir = PROJECT_ROOT / "Results" / dataset_name
results_dir.mkdir(parents=True, exist_ok=True)

level2_output_file = results_dir / f"gpt5_{dataset_name}_level2_concepts.json"

with open(level2_output_file, "w", encoding="utf-8") as f:
    json.dump(refined_level2_concepts, f, indent=2, ensure_ascii=False)

print(f"Saved second-level concepts to: {level2_output_file}")


=== Processing concept 0 ===
→ KEEP

=== Processing concept 1 ===
→ SPLIT

=== Processing concept 2 ===
→ SPLIT

=== Processing concept 3 ===
→ KEEP

=== Processing concept 4 ===
→ KEEP

=== Processing concept 5 ===
→ KEEP

=== Processing concept 6 ===
→ KEEP

=== Processing concept 7 ===
→ KEEP

=== Processing concept 8 ===
→ KEEP

=== Processing concept 9 ===
→ KEEP
Number of refined concepts: 10
Example refined concept:
{'sub_concepts': [{'name': 'Pancreatic acinar secretion', 'genes': ['PNLIP', 'PRSS2', 'CLPS', 'AMY2A', 'CELA2A', 'CPA2', 'PRSS1', 'AMY2B', 'CELA2B', 'RNASE1', 'REG1A', 'SPINK1', 'REG1B', 'REG3G', 'CEL', 'CUZD1', 'REG3A', 'PNLIPRP1', 'PNLIPRP2', 'CLPSL1', 'ZG16', 'ANPEP', 'PRSS8', 'HSPA5', 'ERO1LB', 'NUPR1', 'TXNRD1', 'HERPUD1', 'GNMT', 'RBPJL', 'PTF1A', 'SLC26A3', 'SLC47A1', 'SLC27A6', 'SLC30A10', 'GATM', 'LGALS2', 'RARRES2', 'TXNIP', 'HSPB1', 'EGF', 'SLC2A14', 'KIAA1324', 'LRIG1', 'PLEKHA7', 'SELK', 'PCOLCE2', 'MAMDC2', 'AMBRA1', 'BAIAP2L1', 'APOO', 'CHCHD3', 'C14o

In [11]:
import numpy as np
import pandas as pd

# =========================================================
# Utility function: flatten refined second-level concepts
# =========================================================

def flatten_subconcepts_minimal(refined_results):
    """
    Flatten hierarchical refinement results into a flat list of sub-concepts.

    Parameters
    ----------
    refined_results : list[dict]
        Output from hierarchical concept refinement. Each item contains
        a list of sub_concepts.

    Returns
    -------
    flat_concepts : list[dict]
        Flattened list of second-level concepts.
    parent_ids_for_new : list[int]
        Parent level-1 concept index for each flattened second-level concept.
    """
    flat_concepts = []
    parent_ids_for_new = []

    for parent_idx, refined in enumerate(refined_results):
        sub_concepts = refined.get("sub_concepts", [])

        for sub_concept in sub_concepts:
            flat_concepts.append(
                {
                    "name": sub_concept["name"],
                    "genes": sub_concept["genes"],
                    "parent_idx": parent_idx
                }
            )
            parent_ids_for_new.append(parent_idx)

    return flat_concepts, parent_ids_for_new


# =========================================================
# Utility function: hierarchical assignment
# =========================================================

def hierarchical_assign(
    level1_scores,
    level2_scores,
    parent_ids_for_level2,
    level1_concept_names=None,
    level2_concept_names=None,
):
    """
    Assign each cell using a two-level hierarchy.

    First, each cell is assigned to the best level-1 concept.
    Then, among level-2 concepts whose parent matches that level-1 concept,
    the cell is assigned to the best-scoring sub-concept.

    Parameters
    ----------
    level1_scores : np.ndarray
        Cell-by-level1-concept score matrix.
    level2_scores : np.ndarray
        Cell-by-level2-concept score matrix.
    parent_ids_for_level2 : list[int]
        Parent level-1 concept index for each level-2 concept.
    level1_concept_names : list[str], optional
        Names of level-1 concepts.
    level2_concept_names : list[str], optional
        Names of level-2 concepts.

    Returns
    -------
    pd.DataFrame
        Cell-level hierarchical assignments.
    """
    parent_ids_for_level2 = np.array(parent_ids_for_level2)

    level1_pred_idx = np.argmax(level1_scores, axis=1)

    rows = []

    for cell_idx, parent_idx in enumerate(level1_pred_idx):
        candidate_level2_idx = np.where(parent_ids_for_level2 == parent_idx)[0]

        if len(candidate_level2_idx) == 0:
            best_level2_idx = -1
        else:
            candidate_scores = level2_scores[cell_idx, candidate_level2_idx]
            best_level2_idx = candidate_level2_idx[np.argmax(candidate_scores)]

        level1_name = (
            level1_concept_names[parent_idx]
            if level1_concept_names is not None
            else str(parent_idx)
        )

        level2_name = (
            level2_concept_names[best_level2_idx]
            if level2_concept_names is not None and best_level2_idx >= 0
            else None
        )

        rows.append(
            {
                "cell_index": cell_idx,
                "level1_concept_idx": int(parent_idx),
                "level1_concept_name": level1_name,
                "level2_concept_idx": int(best_level2_idx),
                "level2_concept_name": level2_name,
            }
        )

    return pd.DataFrame(rows)

In [12]:
# =========================================================
# Flatten second-level concepts
# =========================================================

flat_level2_concepts, parent_ids_for_level2 = flatten_subconcepts_minimal(
    refined_level2_concepts
)

print(f"Number of flattened level-2 concepts: {len(flat_level2_concepts)}")
print(f"Example level-2 concept: {flat_level2_concepts[0]}")

Number of flattened level-2 concepts: 16
Example level-2 concept: {'name': 'Pancreatic acinar secretion', 'genes': ['PNLIP', 'PRSS2', 'CLPS', 'AMY2A', 'CELA2A', 'CPA2', 'PRSS1', 'AMY2B', 'CELA2B', 'RNASE1', 'REG1A', 'SPINK1', 'REG1B', 'REG3G', 'CEL', 'CUZD1', 'REG3A', 'PNLIPRP1', 'PNLIPRP2', 'CLPSL1', 'ZG16', 'ANPEP', 'PRSS8', 'HSPA5', 'ERO1LB', 'NUPR1', 'TXNRD1', 'HERPUD1', 'GNMT', 'RBPJL', 'PTF1A', 'SLC26A3', 'SLC47A1', 'SLC27A6', 'SLC30A10', 'GATM', 'LGALS2', 'RARRES2', 'TXNIP', 'HSPB1', 'EGF', 'SLC2A14', 'KIAA1324', 'LRIG1', 'PLEKHA7', 'SELK', 'PCOLCE2', 'MAMDC2', 'AMBRA1', 'BAIAP2L1', 'APOO', 'CHCHD3', 'C14orf2', 'ATP5E', 'ATP5H', 'USMG5', 'ATP5L', 'C19orf24', 'GNB2L1', 'FAM213A', 'MRP63', 'ANKRD30BL', 'RASGEF1B', 'C8orf59', 'PRKCDBP', 'PLEKHA6', 'SEPP1', 'TCEB2', 'H3F3B', 'NBEAL1', 'MTRNR2L1', 'STEAP1B', 'C12orf10', 'TM4SF1', 'TACSTD2', 'KRT7', 'KRT17', 'LCN2', 'GPT2', 'GRB10', 'HEXDC', 'FKBP5', 'PLIN5', 'KIAA0922', 'ARMC9', 'GATA4', 'RAB11FIP3', 'ARRB1', 'UST', 'PLXNA2', 'CBS', 

In [13]:
# =========================================================
# Assign cells to flattened second-level concepts
# =========================================================

top_k = 100

top_k_level2_concepts = truncate_concept_genes(
    flat_level2_concepts,
    top_k=top_k
)

level2_score_matrix, level2_concept_names, predicted_level2_concepts = (
    assign_cells_to_concepts_by_zscore(
        top_k_level2_concepts,
        test_matrix,
        gene_to_index
    )
)

print(f"Level-2 score matrix shape: {level2_score_matrix.shape}")
print(f"Number of level-2 concepts: {len(level2_concept_names)}")
print(f"Example predicted level-2 concepts: {predicted_level2_concepts[:10]}")

Level-2 score matrix shape: (121916, 16)
Number of level-2 concepts: 16
Example predicted level-2 concepts: ['ER stress and inflammation', 'Delta/PP peptide hormones', 'Delta/PP peptide hormones', 'Beta insulin secretion', 'ER stress and inflammation', 'Delta/PP peptide hormones', 'ER stress and inflammation', 'Alpha glucagon excitability', 'Beta insulin secretion', 'ER stress and inflammation']


In [16]:
# =========================================================
# Perform hierarchical cell assignment
# =========================================================

level1_concept_names = [concept["name"] for concept in top_k_level1_concepts]
level2_concept_names = [concept["name"] for concept in top_k_level2_concepts]

df_hierarchical_assign = hierarchical_assign(
    level1_scores=level1_score_matrix,
    level2_scores=level2_score_matrix,
    parent_ids_for_level2=parent_ids_for_level2,
    level1_concept_names=level1_concept_names,
    level2_concept_names=level2_concept_names,
)

df_hierarchical_assign.head()

,cell_index,level1_concept_idx,level1_concept_name,level2_concept_idx,level2_concept_name
0,0,1,Islet endocrine hormones,4,ER stress and inflammation
1,1,1,Islet endocrine hormones,3,Delta/PP peptide hormones
2,2,1,Islet endocrine hormones,3,Delta/PP peptide hormones
3,3,1,Islet endocrine hormones,1,Beta insulin secretion
4,4,1,Islet endocrine hormones,4,ER stress and inflammation


In [17]:
# =========================================================
# Save hierarchical assignment results
# =========================================================

results_dir = PROJECT_ROOT / "Results" / dataset_name
results_dir.mkdir(parents=True, exist_ok=True)

assignment_file = results_dir / f"gpt5_{dataset_name}_hierarchical_assignments.csv"

df_hierarchical_assign.to_csv(assignment_file, index=False)

print(f"Saved hierarchical assignments to: {assignment_file}")

Saved hierarchical assignments to: /home/mcb/users/hchen26/method/textgrad/LLM-ITL-main/scConcept_github/Results/Pancreas_10k/gpt5_Pancreas_10k_hierarchical_assignments.csv


In [19]:
def flatten_labels(labels):
    """
    Convert labels into a flat 1D numpy array.
    """
    labels = np.array(labels, dtype=object).reshape(-1)

    flattened = []
    for label in labels:
        if isinstance(label, np.ndarray):
            if label.size == 1:
                flattened.append(label.item())
            else:
                flattened.append(tuple(label.tolist()))
        else:
            flattened.append(label)

    return np.array(flattened)

In [ ]:
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score

# =========================================================
# Evaluate hierarchical concept assignment
# =========================================================
# Compare predicted second-level concept assignments with
# ground-truth labels.

true_labels = flatten_labels(test_labels)

predicted_labels = df_hierarchical_assign["level2_concept_idx"].values

ari = adjusted_rand_score(true_labels, predicted_labels)
nmi = adjusted_mutual_info_score(true_labels, predicted_labels)

print("ARI:", ari)
print("NMI:", nmi)